In [1]:
# ============================================================
# BDSL47 - Signer-Independent Classification
# Landmark Normalization + Scaling Experiment (Production-Ready)
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. SETTINGS
# ============================================================

DATA_DIR = (
    r"D:\aaa EAAI Major Revision\static one handed\BdSL47_Updated_2026\bdsl47_csv"
)

TRAIN_FILE = os.path.join(DATA_DIR, "train.csv")
VAL_FILE   = os.path.join(DATA_DIR, "validation.csv")
TEST_FILE  = os.path.join(DATA_DIR, "test.csv")

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


# ============================================================
# NORMALIZATION SETTINGS
# ============================================================

APPLY_TRANSLATION_NORMALIZATION = True
APPLY_SCALE_NORMALIZATION = True
APPLY_ROTATION_NORMALIZATION = False


# ============================================================
# 2. LOAD DATA
# ============================================================

train_df = pd.read_csv(TRAIN_FILE)
val_df   = pd.read_csv(VAL_FILE)
test_df  = pd.read_csv(TEST_FILE)

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print(f"Train samples:      {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples:       {len(test_df)}")


# ============================================================
# 3. FEATURES AND LABEL
# ============================================================

FEATURE_COLUMNS = [
    f"{axis}{i:02d}"
    for i in range(21)
    for axis in ["x", "y", "z"]
]

LABEL_COLUMN = "Label"


# ============================================================
# 4. CHECK COLUMNS
# ============================================================

for name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    missing = [
        c for c in FEATURE_COLUMNS + [LABEL_COLUMN]
        if c not in df.columns
    ]
    if missing:
        raise ValueError(
            f"Missing columns in {name}.csv: {missing}"
        )


# ============================================================
# 5. EXTRACT X AND y
# ============================================================

X_train = train_df[FEATURE_COLUMNS].values.astype(np.float32)
y_train = train_df[LABEL_COLUMN].values.astype(np.int64)

X_val = val_df[FEATURE_COLUMNS].values.astype(np.float32)
y_val = val_df[LABEL_COLUMN].values.astype(np.int64)

X_test = test_df[FEATURE_COLUMNS].values.astype(np.float32)
y_test = test_df[LABEL_COLUMN].values.astype(np.int64)


# ============================================================
# 6. BASIC VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("DATA VALIDATION")
print("=" * 70)

if X_train.shape[1] != 63:
    raise ValueError(
        f"Expected 63 features, found {X_train.shape[1]}"
    )

print(f"Number of classes: {len(np.unique(y_train))}")

if np.isnan(X_train).any() or np.isnan(X_val).any() or np.isnan(X_test).any():
    raise ValueError("NaN values found in data.")


# ============================================================
# 7. RESHAPE
# ============================================================

X_train = X_train.reshape(-1, 21, 3)
X_val   = X_val.reshape(-1, 21, 3)
X_test  = X_test.reshape(-1, 21, 3)


# ============================================================
# 8. LANDMARK NORMALIZATION FUNCTIONS
# ============================================================

def normalize_landmarks(X):
    X = X.copy()
    for sample_idx in range(len(X)):
        landmarks = X[sample_idx]
        wrist = landmarks[0].copy()

        if APPLY_TRANSLATION_NORMALIZATION:
            landmarks = landmarks - wrist

        if APPLY_SCALE_NORMALIZATION:
            distances = np.linalg.norm(landmarks, axis=1)
            scale = np.max(distances)
            if scale > 1e-8:
                landmarks = landmarks / scale

        if APPLY_ROTATION_NORMALIZATION:
            index_mcp = landmarks[5]
            middle_mcp = landmarks[9]
            pinky_mcp = landmarks[17]

            axis_u = middle_mcp - wrist
            norm_u = np.linalg.norm(axis_u)
            if norm_u > 1e-8:
                axis_u = axis_u / norm_u
                axis_v = index_mcp - wrist
                axis_v = axis_v - np.dot(axis_v, axis_u) * axis_u
                norm_v = np.linalg.norm(axis_v)
                if norm_v > 1e-8:
                    axis_v = axis_v / norm_v
                    axis_w = np.cross(axis_u, axis_v)
                    norm_w = np.linalg.norm(axis_w)
                    if norm_w > 1e-8:
                        axis_w = axis_w / norm_w
                        axis_v = np.cross(axis_w, axis_u)
                        axis_v = axis_v / np.linalg.norm(axis_v)
                        rotation_matrix = np.column_stack([axis_v, axis_u, axis_w])
                        landmarks = landmarks @ rotation_matrix

        X[sample_idx] = landmarks
    return X


# ============================================================
# 9. APPLY GEOMETRIC NORMALIZATION
# ============================================================

print("\n" + "=" * 70)
print("LANDMARK NORMALIZATION")
print("=" * 70)

X_train = normalize_landmarks(X_train)
X_val   = normalize_landmarks(X_val)
X_test  = normalize_landmarks(X_test)

print("Normalization completed.")


# ============================================================
# 10. FLATTEN BACK TO 63 FEATURES
# ============================================================

X_train = X_train.reshape(-1, 63)
X_val   = X_val.reshape(-1, 63)
X_test  = X_test.reshape(-1, 63)


# ============================================================
# 11. STANDARDIZATION
# ============================================================

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("\n" + "=" * 70)
print("STANDARDIZATION")
print("=" * 70)
print("StandardScaler fitted on TRAINING data only.")


# ============================================================
# 12. MODEL
# ============================================================

def build_model():
    model = Sequential([
        Dense(128, input_shape=(63,)),
        BatchNormalization(),
        LeakyReLU(),
        Dense(128),
        BatchNormalization(),
        LeakyReLU(),
        Dense(64),
        BatchNormalization(),
        LeakyReLU(),
        Dropout(0.33),
        Dense(32),
        BatchNormalization(),
        LeakyReLU(),
        Dropout(0.25),
        Dense(47, activation="softmax")
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_model()


# ============================================================
# 13. MODEL SUMMARY & TRAINING
# ============================================================

model.summary()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=600,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


# ============================================================
# 14. EVALUATION & METRICS
# ============================================================

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")


# ============================================================
# 15. SAVE MODEL, SCALER, AND PIPELINE CONFIGURATION
# ============================================================

model_path = os.path.join(
    DATA_DIR,
    "bdsl47_signer_independent_normalized_mlp.keras"
)
model.save(model_path)

scaler_path = os.path.join(DATA_DIR, "scaler.pkl")
joblib.dump(scaler, scaler_path)

config = {
    "apply_translation_normalization": APPLY_TRANSLATION_NORMALIZATION,
    "apply_scale_normalization": APPLY_SCALE_NORMALIZATION,
    "apply_rotation_normalization": APPLY_ROTATION_NORMALIZATION,
    "feature_columns": FEATURE_COLUMNS,
    "num_classes": 47
}

config_path = os.path.join(DATA_DIR, "pipeline_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("\n" + "=" * 70)
print("PRODUCTION ASSETS SAVED SUCCESSFULLY")
print("=" * 70)
print(f"Model File:  {model_path}")
print(f"Scaler File: {scaler_path}")
print(f"Config File: {config_path}")
print("\nExperiment completed successfully.")

DATASET INFORMATION
Train samples:      28234
Validation samples: 9457
Test samples:       9341

DATA VALIDATION
Number of classes: 47

LANDMARK NORMALIZATION
Normalization completed.

STANDARDIZATION
StandardScaler fitted on TRAINING data only.


c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 47)             │         1,551 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,999 (148.43 KB)

 Trainable params: 37,295 (145.68 KB)

 Non-trainable params: 704 (2.75 KB)

Epoch 1/600
442/442 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.3892 - loss: 2.3179 - val_accuracy: 0.5548 - val_loss: 1.6837
Epoch 2/600
442/442 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6608 - loss: 1.0688 - val_accuracy: 0.6191 - val_loss: 1.4272
Epoch 3/600
442/442 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7501 - loss: 0.7393 - val_accuracy: 0.6463 - val_loss: 1.3847
Epoch 4/600
442/442 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7984 - loss: 0.5938 - val_accuracy: 0.6597 - val_loss: 1.2805
Epoch 5/600
442/442 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8233 - loss: 0.5002 - val_accuracy: 0.6549 - val_loss: 1.4787
Epoch 6/600
442/442 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8478 - loss: 0.4357 - val_accuracy: 0.6944 - val_loss: 1.5091
Epoch 7/600
442/442 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8622 - loss: 0.3923 - val_accuracy: 0.6851 - val_loss: 1.5872
Epoch 8/600
442/442 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8671 - loss: 0.3735 - val_accu